# 02_clean：数据清洗与存储格式对比

本节完成方式 A + 方式 B（Parquet）：

1. 从 `data/stock/` 读取 10 只股票原始 CSV
2. 统一清洗并生成标准化数据表（`date, code, close` 等字段）
3. 保存 `data/clean/stock_clean.csv`
4. 额外保存 `data/clean/stock_clean.parquet`
5. 演示 Parquet 列式读取、Schema 查看、与 CSV 的读取速度和文件体积对比

## 任务 1：读取并清洗股票原始数据，保存 CSV 与 Parquet

说明：

- 读取 `data/stock/stock_*.csv`
- 统一字段名到英文（`date, code, close` 等）并做基础类型转换
- 合并后保存为 `data/clean/stock_clean.csv` 与 `data/clean/stock_clean.parquet`

In [1]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
STOCK_DIR = ROOT / "data" / "stock"
CLEAN_DIR = ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

stock_files = sorted(STOCK_DIR.glob("stock_*.csv"))
if not stock_files:
    raise FileNotFoundError("未找到 data/stock/stock_*.csv 文件")

frames = []
for fp in stock_files:
    df = pd.read_csv(fp)

    # 统一字段并保留后续分析常用列
    rename_map = {
        "日期": "date",
        "代码": "code",
        "名称": "name",
        "行业": "industry",
        "开盘价": "open",
        "收盘价": "close",
        "最高价": "high",
        "最低价": "low",
        "成交量": "volume",
        "成交额": "amount",
    }
    df = df.rename(columns=rename_map)

    required = ["date", "code", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"跳过 {fp.name}，缺少字段: {missing}")
        continue

    # 类型标准化
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["code"] = df["code"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)

    for col in ["open", "close", "high", "low", "volume", "amount"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 基础清洗：去重、去无效日期
    df = df.dropna(subset=["date"]).drop_duplicates(subset=["date", "code"]).copy()
    frames.append(df)

stock_clean = pd.concat(frames, ignore_index=True)
stock_clean = stock_clean.sort_values(["code", "date"]).reset_index(drop=True)

# 计算日对数收益率（便于后续分析）
stock_clean["log_ret"] = stock_clean.groupby("code")["close"].transform(
    lambda s: np.log(s / s.shift(1))
)

# 保存 CSV 和 Parquet（方式 A + 方式 B）
csv_path = CLEAN_DIR / "stock_clean.csv"
parquet_path = CLEAN_DIR / "stock_clean.parquet"

stock_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")
stock_clean.to_parquet(parquet_path, index=False)

print(f"stock_clean shape: {stock_clean.shape}")
print(f"CSV 已保存: {csv_path}")
print(f"Parquet 已保存: {parquet_path}")
stock_clean.head(5)

stock_clean shape: (14590, 11)
CSV 已保存: /Users/yijun/YJ/coding-workplace/ex_P02/dshw-p01/data/clean/stock_clean.csv
Parquet 已保存: /Users/yijun/YJ/coding-workplace/ex_P02/dshw-p01/data/clean/stock_clean.parquet


,date,open,close,high,low,volume,amount,code,name,industry,log_ret
0,2020-01-02,4530.67,4497.51,4641.17,4490.61,101213040.0,3.342374e+09,000002,万科A,房地产,NaN
1,2020-01-03,4518.23,4427.07,4532.05,4389.77,80553629.0,2.584310e+09,000002,万科A,房地产,-0.015786
2,2020-01-06,4385.63,4352.48,4387.01,4316.56,87684058.0,2.761449e+09,000002,万科A,房地产,-0.016992
3,2020-01-07,4366.29,4387.01,4410.49,4330.38,57793343.0,1.827511e+09,000002,万科A,房地产,0.007902
4,2020-01-08,4323.47,4375.96,4388.39,4288.94,52999684.0,1.667144e+09,000002,万科A,房地产,-0.002522


## 任务 2：演示 Parquet 的列式读取、Schema 与性能对比

本节按作业要求展示三点：

- 列式读取（只加载 `date, code, close`）
- Schema（类型契约）查看
- CSV 与 Parquet 在读取耗时、文件体积上的对比

In [2]:
csv_file = "data/clean/stock_clean.csv"
parquet_file = "data/clean/stock_clean.parquet"

# 列式读取（只加载需要列）
df_col = pd.read_parquet(parquet_file, columns=["date", "code", "close"])
print("Parquet 列式读取示例（前5行）：")
print(df_col.head())

# 查看 Schema（类型契约）
schema = pq.read_schema(parquet_file)
print("\nParquet Schema:")
print(schema)

# CSV 与 Parquet 读取速度对比
t0 = time.time()
_ = pd.read_csv(csv_file)
csv_time = time.time() - t0


t0 = time.time()
_ = pd.read_parquet(parquet_file)
parquet_time = time.time() - t0

csv_size_kb = os.path.getsize(csv_file) / 1024
parquet_size_kb = os.path.getsize(parquet_file) / 1024

print("\n读取与体积对比：")
print(f"CSV      读取耗时: {csv_time:.4f}s   文件大小: {csv_size_kb:.1f} KB")
print(f"Parquet  读取耗时: {parquet_time:.4f}s   文件大小: {parquet_size_kb:.1f} KB")

# 给结论单元传递结果
perf_summary = {
    "csv_time": csv_time,
    "parquet_time": parquet_time,
    "csv_size_kb": csv_size_kb,
    "parquet_size_kb": parquet_size_kb,
}
perf_summary

Parquet 列式读取示例（前5行）：
        date    code    close
0 2020-01-02  000002  4497.51
1 2020-01-03  000002  4427.07
2 2020-01-06  000002  4352.48
3 2020-01-07  000002  4387.01
4 2020-01-08  000002  4375.96

Parquet Schema:
date: timestamp[ns]
open: double
close: double
high: double
low: double
volume: double
amount: double
code: string
name: string
industry: string
log_ret: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1301

读取与体积对比：
CSV      读取耗时: 0.0200s   文件大小: 1554.2 KB
Parquet  读取耗时: 0.0031s   文件大小: 675.5 KB


{'csv_time': 0.020025253295898438,
 'parquet_time': 0.0030798912048339844,
 'csv_size_kb': 1554.2265625,
 'parquet_size_kb': 675.51953125}

## 结果解读

基于本次实测结果：

- 读取耗时：CSV 约 `0.0200s`，Parquet 约 `0.0031s`，Parquet 更快。
- 文件体积：CSV 约 `1554.2 KB`，Parquet 约 `675.5 KB`，Parquet 明显更小。

结论：

- 在本次数据规模下，CSV 与 Parquet 的差异已经比较明显，尤其是文件体积与读取速度。
- 当数据规模进一步扩大（例如百万行以上）、字段更多、且分析时经常只读取部分列时，Parquet 的优势会更显著（列式存储 + 压缩效率更高）。
- 课程作业中可采用“CSV + Parquet 并存”策略：CSV 保持通用可读，Parquet 用于高效分析。

## 第三部分：数据清洗

以下内容对应作业第 3 部分要求：

- 3.1 单表清洗（缺失值、类型、重复值、离群值）
- 3.2 宽表与长表转换
- 3.3 多表合并（个股 + 指数 + 宏观）

每个步骤都给出文字说明，并在代码中输出清洗前后对比结果。

### 3.1 单表清洗（对每只股票）

本节按以下顺序清洗，并输出“清洗前后”变化：

1. 缺失值检测：统计各列缺失数量与比例，分析可能来源（停牌、非交易日、数据源缺口）
2. 缺失值处理：对行情字段按股票代码分组做时间序列前向填充（`ffill`）
3. 日期格式统一：转换为 `datetime64`，并在处理中以日期为索引
4. 数据类型检查：将价格与成交量字段统一为数值型
5. 重复值处理：按 `code + date` 去重
6. 离群值标注：计算日收益率，标记 `|return| > 20%` 为 `is_extreme=True`（不删除）

In [3]:
# 读取原始个股数据（按 stock_*.csv）
raw_files = sorted((ROOT / "data" / "stock").glob("stock_*.csv"))
raw_frames = []
for fp in raw_files:
    tmp = pd.read_csv(fp)
    tmp = tmp.rename(
        columns={
            "日期": "date",
            "代码": "code",
            "名称": "name",
            "行业": "industry",
            "开盘价": "open",
            "收盘价": "close",
            "最高价": "high",
            "最低价": "low",
            "成交量": "volume",
            "成交额": "amount",
        }
    )
    raw_frames.append(tmp)

raw_stock = pd.concat(raw_frames, ignore_index=True)
raw_stock["code"] = raw_stock["code"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)

print(f"原始数据行数: {len(raw_stock):,}")
print(f"原始数据列: {list(raw_stock.columns)}")

# 1) 缺失值检测（清洗前）
missing_before = pd.DataFrame({
    "missing_count": raw_stock.isna().sum(),
    "missing_ratio": raw_stock.isna().mean().round(4),
}).sort_values("missing_count", ascending=False)
print("\n[清洗前] 各列缺失情况：")
display(missing_before)

# 2) 日期与类型标准化
work = raw_stock.copy()
work["date"] = pd.to_datetime(work["date"], errors="coerce")

num_cols = ["open", "close", "high", "low", "volume", "amount"]
for c in num_cols:
    if c in work.columns:
        work[c] = pd.to_numeric(work[c], errors="coerce")

# 3) 重复值处理前先计数
dup_before = work.duplicated(subset=["code", "date"]).sum()

# 4) 缺失值处理：对时间序列字段按股票分组后 ffill
work = work.sort_values(["code", "date"]).set_index("date")
work[num_cols] = work.groupby("code")[num_cols].ffill()
work = work.reset_index()

# 5) 删除关键键值缺失（date/code/close）
before_drop = len(work)
work = work.dropna(subset=["date", "code", "close"]).copy()
after_drop = len(work)

# 6) 去重
work = work.drop_duplicates(subset=["code", "date"]).copy()
dup_after = work.duplicated(subset=["code", "date"]).sum()

# 7) 离群值标注：单日涨跌幅绝对值 > 20%
work = work.sort_values(["code", "date"]).copy()
work["daily_ret"] = work.groupby("code")["close"].pct_change()
work["is_extreme"] = work["daily_ret"].abs() > 0.20
extreme_count = int(work["is_extreme"].sum())

# 清洗后缺失值统计
missing_after = pd.DataFrame({
    "missing_count": work.isna().sum(),
    "missing_ratio": work.isna().mean().round(4),
}).sort_values("missing_count", ascending=False)

print("\n[清洗后] 各列缺失情况：")
display(missing_after)

print("\n清洗前后变化汇总：")
print(f"- 原始行数: {len(raw_stock):,}")
print(f"- 去除关键字段缺失导致减少: {before_drop - after_drop:,} 行")
print(f"- 重复记录（code+date）清洗前: {dup_before:,}，清洗后: {dup_after:,}")
print(f"- 离群值标注数量（|daily_ret|>20%）: {extreme_count:,}")

# 覆盖写回 clean 数据（方式A+方式B保持一致）
stock_clean = work.copy()
stock_clean.to_csv(ROOT / "data" / "clean" / "stock_clean.csv", index=False, encoding="utf-8-sig")
stock_clean.to_parquet(ROOT / "data" / "clean" / "stock_clean.parquet", index=False)

print("\n已更新: data/clean/stock_clean.csv 与 data/clean/stock_clean.parquet")
stock_clean.head(5)

原始数据行数: 14,590
原始数据列: ['date', 'open', 'close', 'high', 'low', 'volume', 'amount', 'code', 'name', 'industry']

[清洗前] 各列缺失情况：


,missing_count,missing_ratio
date,0,0.0
open,0,0.0
close,0,0.0
high,0,0.0
low,0,0.0
volume,0,0.0
amount,0,0.0
code,0,0.0
name,0,0.0
industry,0,0.0



[清洗后] 各列缺失情况：


,missing_count,missing_ratio
daily_ret,10,0.0007
date,0,0.0000
open,0,0.0000
close,0,0.0000
high,0,0.0000
low,0,0.0000
volume,0,0.0000
amount,0,0.0000
code,0,0.0000
name,0,0.0000



清洗前后变化汇总：
- 原始行数: 14,590
- 去除关键字段缺失导致减少: 0 行
- 重复记录（code+date）清洗前: 0，清洗后: 0
- 离群值标注数量（|daily_ret|>20%）: 0

已更新: data/clean/stock_clean.csv 与 data/clean/stock_clean.parquet


,date,open,close,high,low,volume,amount,code,name,industry,daily_ret,is_extreme
0,2020-01-02,4530.67,4497.51,4641.17,4490.61,101213040.0,3.342374e+09,000002,万科A,房地产,NaN,False
1,2020-01-03,4518.23,4427.07,4532.05,4389.77,80553629.0,2.584310e+09,000002,万科A,房地产,-0.015662,False
2,2020-01-06,4385.63,4352.48,4387.01,4316.56,87684058.0,2.761449e+09,000002,万科A,房地产,-0.016849,False
3,2020-01-07,4366.29,4387.01,4410.49,4330.38,57793343.0,1.827511e+09,000002,万科A,房地产,0.007933,False
4,2020-01-08,4323.47,4375.96,4388.39,4288.94,52999684.0,1.667144e+09,000002,万科A,房地产,-0.002519,False


### 3.2 宽表与长表转换

说明：

- 宽表（Wide）：以日期为索引、每只股票一列收盘价，适合做相关系数、协方差、矩阵运算、时间序列联动分析。
- 长表（Long）：每行一条观测（`date, code, close`），适合分组统计、回归建模、可视化分面和多表合并。

In [4]:
# 以清洗后的 stock_clean 为基础构造宽表与长表
close_long = stock_clean[["date", "code", "close"]].dropna().copy()

# 宽表：index=date, columns=code, values=close
close_wide = close_long.pivot_table(index="date", columns="code", values="close", aggfunc="last").sort_index()

# 再转回长表
close_long_from_wide = (
    close_wide.reset_index()
    .melt(id_vars="date", var_name="code", value_name="close")
    .dropna(subset=["close"])
    .sort_values(["code", "date"])
    .reset_index(drop=True)
)

# 保存中间结果
close_wide.to_csv(ROOT / "data" / "clean" / "close_wide.csv", encoding="utf-8-sig")
close_long_from_wide.to_csv(ROOT / "data" / "clean" / "close_long_from_wide.csv", index=False, encoding="utf-8-sig")

print(f"宽表 shape: {close_wide.shape} (行=交易日, 列=股票代码)")
print(f"长表 shape: {close_long_from_wide.shape}")

print("\n宽表预览：")
display(close_wide.head())

print("\n长表预览：")
display(close_long_from_wide.head())

宽表 shape: (1515, 10) (行=交易日, 列=股票代码)
长表 shape: (14590, 3)

宽表预览：


code,000002,000063,002352,002594,600036,600519,600938,601238,601398,601857
date,,,,,,,,,,
2020-01-02,4497.51,602.59,129.54,49.09,193.11,8495.30,NaN,19.71,10.64,7.88
2020-01-03,4427.07,621.29,128.53,48.96,195.69,8108.58,NaN,20.21,10.67,7.99
2020-01-06,4352.48,623.84,127.36,49.20,194.90,8104.29,NaN,19.70,10.64,8.36
2020-01-07,4387.01,622.14,130.30,48.97,194.45,8228.64,NaN,20.09,10.71,8.20
2020-01-08,4375.96,607.01,128.74,48.19,190.77,8180.60,NaN,20.12,10.53,8.36



长表预览：


,date,code,close
0,2020-01-02,000002,4497.51
1,2020-01-03,000002,4427.07
2,2020-01-06,000002,4352.48
3,2020-01-07,000002,4387.01
4,2020-01-08,000002,4375.96


### 3.3 多表合并（个股 + 指数 + 宏观）

说明：

- 先将个股日度数据与指数数据按 `date` 做 `left join`。
- 再将月度宏观数据映射到日频：提取 `month=YYYY-MM` 后按月份合并。
- 每一步都记录合并前后行数，解释行数变化原因：
  - `left join` 下主表行数通常不变；
  - 行数变化多由主表去重、日期解析失败或键值缺失引起。

In [5]:
# 读取指数数据
idx300 = pd.read_csv(ROOT / "data" / "index" / "index_000300.csv")
idx300 = idx300.rename(columns={"日期": "date", "收盘价": "hs300_close"})[["date", "hs300_close"]]
idx300["date"] = pd.to_datetime(idx300["date"], errors="coerce")
idx300 = idx300.dropna(subset=["date"]).drop_duplicates(subset=["date"])

idx905 = pd.read_csv(ROOT / "data" / "index" / "index_000905.csv")
idx905 = idx905.rename(columns={"日期": "date", "收盘价": "zz500_close"})[["date", "zz500_close"]]
idx905["date"] = pd.to_datetime(idx905["date"], errors="coerce")
idx905 = idx905.dropna(subset=["date"]).drop_duplicates(subset=["date"])

# 读取宏观数据并映射到 month
macro_cpi = pd.read_csv(ROOT / "data" / "macro" / "macro_cpi.csv")
macro_m2 = pd.read_csv(ROOT / "data" / "macro" / "macro_m2_yoy.csv")


def parse_macro(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    date_col = next((c for c in ["日期", "date", "月份", "时间"] if c in df.columns), None)
    if date_col is None:
        return pd.DataFrame(columns=["month", value_name])

    # 兼容常见列名
    cand = [value_name, "同比", "m2同比", "M2同比", "value"]
    val_col = next((c for c in cand if c in df.columns), None)
    if val_col is None:
        non_date = [c for c in df.columns if c != date_col]
        for c in non_date:
            if pd.to_numeric(df[c], errors="coerce").notna().sum() > 0:
                val_col = c
                break
    if val_col is None:
        return pd.DataFrame(columns=["month", value_name])

    out = df[[date_col, val_col]].copy()
    out.columns = ["date", value_name]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date"])
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    out["month"] = out["date"].dt.to_period("M").astype(str)

    # 同一月份保留最后一期
    out = out.sort_values("date").drop_duplicates(subset=["month"], keep="last")
    return out[["month", value_name]]


cpi_month = parse_macro(macro_cpi, "cpi_yoy")
m2_month = parse_macro(macro_m2, "m2_yoy")

# 多表合并：主表为 stock_clean（left join）
combined = stock_clean.copy()
row_before = len(combined)

combined = combined.merge(idx300, on="date", how="left")
row_after_idx300 = len(combined)

combined = combined.merge(idx905, on="date", how="left")
row_after_idx905 = len(combined)

combined["month"] = combined["date"].dt.to_period("M").astype(str)
combined = combined.merge(cpi_month, on="month", how="left")
row_after_cpi = len(combined)

combined = combined.merge(m2_month, on="month", how="left")
row_after_m2 = len(combined)

# 保存综合数据
combined_out = combined.sort_values(["code", "date"]).reset_index(drop=True)
combined_out.to_csv(ROOT / "data" / "combined" / "combined_data.csv", index=False, encoding="utf-8-sig")

print("合并行数变化：")
print(f"- 主表（stock_clean）行数: {row_before:,}")
print(f"- 合并 hs300 后行数: {row_after_idx300:,}")
print(f"- 合并 zz500 后行数: {row_after_idx905:,}")
print(f"- 合并 cpi(月映射)后行数: {row_after_cpi:,}")
print(f"- 合并 m2(月映射)后行数: {row_after_m2:,}")

print("\n说明：本流程使用 left join，主表行数通常保持不变；新增的是外部字段而非新增行。")
print("已保存：data/combined/combined_data.csv")

combined_out.head(5)

合并行数变化：
- 主表（stock_clean）行数: 14,590
- 合并 hs300 后行数: 14,590
- 合并 zz500 后行数: 14,590
- 合并 cpi(月映射)后行数: 14,590
- 合并 m2(月映射)后行数: 14,590

说明：本流程使用 left join，主表行数通常保持不变；新增的是外部字段而非新增行。
已保存：data/combined/combined_data.csv


/var/folders/0_/nxm0n2m16mv8z7z9m2h2jf640000gn/T/ipykernel_42629/723451044.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["date"] = pd.to_datetime(out["date"], errors="coerce")


,date,open,close,high,low,volume,amount,code,name,industry,daily_ret,is_extreme,hs300_close,zz500_close,month,cpi_yoy,m2_yoy
0,2020-01-02,4530.67,4497.51,4641.17,4490.61,101213040.0,3.342374e+09,000002,万科A,房地产,NaN,False,4152.241,5366.138,2020-01,4.5,NaN
1,2020-01-03,4518.23,4427.07,4532.05,4389.77,80553629.0,2.584310e+09,000002,万科A,房地产,-0.015662,False,4144.965,5380.637,2020-01,4.5,NaN
2,2020-01-06,4385.63,4352.48,4387.01,4316.56,87684058.0,2.761449e+09,000002,万科A,房地产,-0.016849,False,4129.295,5434.850,2020-01,4.5,NaN
3,2020-01-07,4366.29,4387.01,4410.49,4330.38,57793343.0,1.827511e+09,000002,万科A,房地产,0.007933,False,4160.227,5499.840,2020-01,4.5,NaN
4,2020-01-08,4323.47,4375.96,4388.39,4288.94,52999684.0,1.667144e+09,000002,万科A,房地产,-0.002519,False,4112.317,5423.797,2020-01,4.5,NaN
